# 📸 Instagram Analiz Uygulaması

Bu proje, **Instaloader** kütüphanesi kullanarak belirlediğiniz bir Instagram hesabının takipçi sayısı, takip edilen sayısı ve son gönderi tarihi gibi verilerini çekerek **Tkinter** ile oluşturduğumuz bir tabloda sunan masaüstü uygulamasıdır.

Aşağıdaki hücreleri **sırasıyla (Shift + Enter)** çalıştırarak kodun nasıl adım adım inşa edildiğini inceleyebilirsiniz.

## 1. Adım: Gerekli Kütüphanelerin İçe Aktarılması

Bu projede kullanacağımız kütüphanelerin her birinin özel bir görevi vardır:

* **`instaloader`**: Instagram verilerini çekmek için kullandığımız araçtır.
* **`tkinter`**: Python'un standart görsel arayüz oluşturucusudur. Pencereler, butonlar ve veri giriş alanları oluşturmak için kullanırız.
* **`ttk` ve `messagebox`**: Tablo (`Treeview`) oluşturmak ve hatalı girişlerde ekrana uyarı penceresi çıkarmak için.

In [ ]:
import instaloader
import tkinter as tk
from tkinter import ttk, messagebox

## 2. Adım: Arka Planda Veri Çeken Fonksiyonlar

Instagram uygulamasındaki bir profilin istatistiklerini elde etmek için iki fonksiyona ihtiyacımız var. Biri kullanıcının temel bilgilerini toplarken diğeri o kullanıcının son gönderisinin ne zaman atıldığını hesaplar.

### 2.1 Son Gönderi Tarihini Bulan Fonksiyon

Bir hesabın ne kadar aktif olduğunu anlamak için son gönderisinin tarihini bulmak işe yarar. `instaloader` doğrudan bu veriyi vermez, bu yüzden profildeki gönderileri döngüye alarak en yeni gönderiyi manuel olarak buluruz.

In [ ]:
def get_last_post_date(profile):
    last_post = None

    # Profildeki tüm gönderileri döngüye alıyoruz
    for post in profile.get_posts():
        # İlk gönderiyi veya tarihi daha yeni olan bir gönderiyi last_post olarak güncelliyoruz
        if not last_post or post.date_utc > last_post.date_utc:
            last_post = post
            
    # Eğer hesabın en az 1 gönderisi varsa tarihini YYYY-MM-DD HH:MM:SS formatında döndürüyoruz
    if last_post:
        return last_post.date_utc.strftime("%Y-%m-%d %H:%M:%S")
    
    # Hiç gönderisi yoksa
    return "Gönderi Yok"

### 2.2 Genel Kullanıcı Bilgilerini Çeken Ana Fonksiyon

Ana işlevi yapan bu fonksiyondur. Arayüzden kullanıcının girdiği ismi alır ve Instaloader'a gönderir. Olası yanlış profil ismi girişinde programın çökmemesi için süreci **`try-except`** bloğunda yürütürüz.

In [ ]:
def get_user_info(username):
    bot = instaloader.Instaloader()
    
    try:
        profile = instaloader.Profile.from_username(bot.context, username)
    except instaloader.exceptions.ProfileNotExistsException:
        return "Girdiğiniz kullanıcı bulunamadı."
    except Exception as e:
        return f"Bir hata oluştu: {e}"

    # Tüm bilgileri bir sözlük (dictionary) olarak saklıyoruz.
    user_info = {
        "Username" : profile.username,
        "Followers" : profile.followers,
        "Followees" : profile.followees,
        "Post Count" : profile.mediacount,
        "Last Post Date" : get_last_post_date(profile) # Yukarıdaki fonksiyonumuzu çağırdık
    }

    return user_info

## 3. Adım: Arayüz ile Arka Planı Birleştirmek

Bu fonksiyon, kullanıcı "Bilgileri Görüntüle" butonuna bastığında çalışır. `get_user_info` fonksiyonundan dönen değer bir sözlük (başarılı) ise bunu tabloya yazar. Dönen değer bir String ise (hata yakalandıysa) uyarı pop-up'ı gösterir.

In [ ]:
def show_user():
    # Arayüzdeki yazı alanından ismi alırız
    username = entry_username.get()
    
    user_info = get_user_info(username)

    if isinstance(user_info, dict):
        # Tablodaki eski verileri temizliyoruz
        for widget in tree.get_children():
            tree.delete(widget)
            
        # Tabloya yeni çekilen kullanıcı verilerini ekliyoruz
        tree.insert("","end",values=(
            user_info["Username"],
            user_info["Followers"],
            user_info["Followees"],
            user_info["Post Count"],
            user_info["Last Post Date"]
        ))
    else:
        # Hata mesajı gönder
        messagebox.showerror("Hata", user_info)


## 4. Adım: Kullanıcı Arayüzünün (GUI) Kurulması

Bu adımda penceremizi oluşturuyoruz ve içine veri giriş kutusunu, butonu ve `Treeview` tablomuzu yerleştiriyoruz.

**NOT:** Bu hücreyi çalıştırdığınızda bilgisayarınızda bir pencere açılacak ve uygulamanız tamamen çalışır hale gelecektir. Notebook bu hücrede asılı (çalışır durumda) kalacaktır.

In [ ]:
# İşlem bütünlüğünü sağlamak için show_user fonksiyonunu eksiksiz tanımlıyoruz
def show_user():
    username = entry_username.get()
    user_info = get_user_info(username)

    if isinstance(user_info, dict):
        for widget in tree.get_children():
            tree.delete(widget)
            
        tree.insert("","end",values=(
            user_info["Username"],
            user_info["Followers"],
            user_info["Followees"],
            user_info["Post Count"],
            user_info["Last Post Date"]
        ))
    else:
        messagebox.showerror("Hata", user_info)

# ----------------- ARAYÜZ KODLARI -----------------
root = tk.Tk()
root.title("Instagram Kullanıcı Bilgi Görüntüleyicisi")

frame = tk.Frame(root)
frame.pack(padx=20, pady=20)

label = tk.Label(frame, text="Kullanıcı Adı: ")
label.grid(row=0, column=0, padx=5, pady=5)

entry_username = tk.Entry(frame)
entry_username.grid(row=0, column=1, padx=5, pady=5)

search_button = tk.Button(frame, text="Bilgileri Görüntüle", command=show_user)
search_button.grid(row=0, column=2, padx=5, pady=5)

tree = ttk.Treeview(root, columns=("Username","Followers","Followees","Post Count","Last Post Date"))
tree.heading("Username",text="Kullanıcı adı")
tree.heading("Followers", text="Takipçiler")
tree.heading("Followees",text="Takip Edilenler")
tree.heading("Post Count", text="Gönderi Sayısı")
tree.heading("Last Post Date",text="Son Gönderi Tarihi")
tree.pack(padx=20, pady=20)

root.mainloop()

---

# 🔄 Programın Çalışma Mantığı ve Kodların Gerçek İşleyiş Sırası

Python kodu sırayla okusa da işleyiş her zaman yukarıdan aşağıya doğru değildir. Aşağıda programı tek parça çalıştırdığınızda arka planda işlemlerin hangi satırlardan geçerek nasıl zıpladığını ve mantığını görebilirsiniz:

### Aşama 1: Arayüzün Çizilmesi ve Bekleme
Python en başta yukarıdaki `show_user`, `get_user_info` ve `get_last_post_date` fonksiyonlarını sadece hafızasına alır ama içine girip çalıştırmaz (çünkü henüz kimse onları çağırmamıştır). Doğrudan arayüz kodlarını okumaya başlar. Pencereyi, giriş kutusunu, butonu ve tabloyu ekler, sonra `mainloop()` sayesinde program **bekleme/dinleme moduna** geçer.

In [ ]:
root = tk.Tk() # 1- Ana pencere yaratıldı
entry_username = tk.Entry(...) # 2- Kullanıcı veri alanları oluşturuldu
search_button = tk.Button(..., command=show_user) # 3- Buton eklendi ve görevi (show_user) atandı
tree = ttk.Treeview(...) # 4- Tablo eklendi

root.mainloop() # 5- PROGRAM DURDU. Kullanıcının butona tıklaması bekleniyor.

### Aşama 2: Kullanıcı Tıklar ve Fonksiyona Zıplanır
Kullanıcı ismini yazıp 'Bilgileri Görüntüle' butonuna bastığında, butona arayüz aşamasında atadığımız `command=show_user` yetkisi devreye girer. Program döngüden çıkar ve hemen o fonksiyona zıplar.

In [ ]:
def show_user():
    # 6- Zıplama gerçekleşti! Python artık burada.
    # 7- Kullanıcının arayüze yazdığı isim alınıyor
    username = entry_username.get()
    
    # 8- Asıl verileri çekmek için get_user_info fonksiyonuna zıplanıyor!
    user_info = get_user_info(username)

### Aşama 3: Verilerin Çekilmesi ve Son Gönderi Fonksiyonuna Geçiş
Python, `get_user_info` fonksiyonuna gelir ve arka planda Instagram verilerini çekerken, içindeki son bir görev için 3. bir fonksiyona (`get_last_post_date`) daha zıplar.

In [ ]:
def get_user_info(username):
    bot = instaloader.Instaloader()
    try:
        # 9- Instaloader ile profile bağlanılır
        profile = instaloader.Profile.from_username(bot.context, username)
    except instaloader.exceptions.ProfileNotExistsException:
        return "Girdiğiniz kullanıcı bulunamadı." # Eğer hata olursa hemen 15. adıma geri fırlar

    # 10- Başarılı ise sözlük oluşturulmaya başlanır
    user_info = {
        "Username" : profile.username,
        "Followers" : profile.followers,
        "Followees" : profile.followees,
        "Post Count" : profile.mediacount,
        # 11- Son gönderi tarihi için get_last_post_date fonksiyonuna zıplanır!
        "Last Post Date" : get_last_post_date(profile) 
    }

    # 14- Toplanan sözlük show_user fonksiyonuna geri gönderilir
    return user_info

def get_last_post_date(profile):
    # 12- Tüm gönderiler incelenir ve en son gönderi bulunur
    # ... (tarih hesaplanır) ...
    
    # 13- Bulunan tarih formatlanıp 11. adımdaki sözlüğün içine geri gönderilir
    return last_post.date_utc.strftime("%Y-%m-%d %H:%M:%S")

### Aşama 4: Verilerin Tabloya Yazdırılması ve Bekleme Modu
Toplanan tüm veriler başarıyla `show_user` fonksiyonuna döndüğünde, fonksiyon tabloyu (`Treeview`) günceller.

In [ ]:
def show_user():
    # ...
    # 15- Veriler geldi! Şimdi başarılı mı başarısız mı kontrol ediliyor.
    if isinstance(user_info, dict):
        # 16- Başarılıysa tablo temizlenip veriler eklenir
        for widget in tree.get_children():
            tree.delete(widget)
            
        tree.insert("","end",values=(
            # ... değerler ...
        ))
    else:
        # 16- Eğer başarısız olduysa hata mesajı çıkar
        messagebox.showerror("Hata", user_info)

    # 17- FONKSİYON BİTTİ.

Fonksiyon işini tamamen bitirdiğinde, Python programı başı boş bırakmaz. En baştaki Aşama 1'de beklemeye aldığı o kilit satıra, yani `root.mainloop()` satırına sessizce geri döner ve kullanıcının yeni bir kullanıcı araması ihtimaline karşı arayüzü tekrar **dinleme moduna** geçirir.